# ResNet50 Training Refinements

This notebook starts from the evaluated ResNet50 baseline and tests a
more disciplined fine-tuning recipe: stronger augmentation, label
smoothing, AdamW, learning-rate scheduling, and early stopping. It is
intentionally separate from the baseline notebook so improvements can
be attributed to training changes rather than evaluation changes.

## 1. Experiment Plan

| Experiment | Change | Purpose |
| --- | --- | --- |
| FT-V2 | longer fine-tuning + early stopping | test whether the baseline is undertrained |
| FT-V3 | scheduler + AdamW | stabilize deeper fine-tuning |
| FT-V4 | stronger augmentation + label smoothing | improve robustness to plating, lighting, and label noise |

The final evaluation should reuse the same top-1, top-5, class report,
confusion-pair, and qualitative-error protocol from notebook 1.

In [ ]:
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

In [ ]:
@dataclass(frozen=True)
class CFG:
    """Configuration for ResNet50 refinement experiments."""

    SEED: int = 42
    BATCH_SIZE: int = 32
    IMAGE_SIZE: tuple[int, int] = (224, 224)
    NUM_CLASSES: int = 101
    NUM_WORKERS: int = 2
    MAX_EPOCHS: int = 15
    PATIENCE: int = 3
    LEARNING_RATE: float = 3e-5
    WEIGHT_DECAY: float = 1e-4
    LABEL_SMOOTHING: float = 0.05
    TOP_K: int = 5
    TRAINABLE_PREFIXES: tuple[str, ...] = ("layer3", "layer4", "fc")
    DATA_DIR: Path = Path("/kaggle/input/datasets/kmader/food41")
    BASELINE_ARTIFACT_DIR: Path = Path("/kaggle/input/food101-baseline-artifacts")
    RESULTS_DIR: Path = Path("/kaggle/working/results/resnet50_refinements")
    BASELINE_CHECKPOINT_NAME: str = "finetuned_exp_2_layer3_4.pth"


CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def create_data_manifest(image_dir: Path) -> pd.DataFrame:
    """Create an image-path and label manifest from class folders."""
    records = []
    for class_dir in sorted(path for path in image_dir.iterdir() if path.is_dir()):
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                records.append({"path": str(image_path), "label": class_dir.name})
    return pd.DataFrame.from_records(records)


IMAGE_DIR = CFG.DATA_DIR / "images"
df = create_data_manifest(IMAGE_DIR)
class_names = sorted(df["label"].unique())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=CFG.SEED,
    stratify=df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=CFG.SEED,
    stratify=temp_df["label"],
)
print(len(train_df), len(val_df), len(test_df))

In [ ]:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

TRAIN_TRANSFORMS = transforms.Compose(
    [
        transforms.RandomResizedCrop(CFG.IMAGE_SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply(
            [transforms.ColorJitter(0.2, 0.2, 0.2, 0.05)],
            p=0.6,
        ),
        transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)

VAL_TRANSFORMS = transforms.Compose(
    [
        transforms.Resize(CFG.IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)

In [ ]:
class FoodDataset(Dataset):
    """Food-101 dataset wrapper."""

    def __init__(self, dataframe, class_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, self.class_to_idx[row["label"]]


train_loader = DataLoader(
    FoodDataset(train_df, class_to_idx, TRAIN_TRANSFORMS),
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
val_loader = DataLoader(
    FoodDataset(val_df, class_to_idx, VAL_TRANSFORMS),
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    FoodDataset(test_df, class_to_idx, VAL_TRANSFORMS),
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=device.type == "cuda",
)

In [ ]:
def make_classifier_head(in_features: int) -> nn.Sequential:
    """Create the Food-101 classifier head."""
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, CFG.NUM_CLASSES),
    )


def build_resnet50() -> nn.Module:
    """Create ResNet50 with the project classifier head."""
    model = models.resnet50(weights=None)
    model.fc = make_classifier_head(model.fc.in_features)
    return model


def resolve_baseline_checkpoint() -> Path:
    """Find the baseline checkpoint from attached Kaggle artifacts."""
    candidates = [
        CFG.BASELINE_ARTIFACT_DIR / CFG.BASELINE_CHECKPOINT_NAME,
        CFG.BASELINE_ARTIFACT_DIR / "results" / CFG.BASELINE_CHECKPOINT_NAME,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = sorted(CFG.BASELINE_ARTIFACT_DIR.rglob(CFG.BASELINE_CHECKPOINT_NAME))
    if matches:
        return matches[0]
    raise FileNotFoundError("Attach the baseline checkpoint artifact dataset first.")


def configure_trainable_layers(model: nn.Module) -> None:
    """Freeze all layers except selected fine-tuning prefixes."""
    for name, parameter in model.named_parameters():
        parameter.requires_grad = name.startswith(CFG.TRAINABLE_PREFIXES)

In [ ]:
model = build_resnet50()
checkpoint = resolve_baseline_checkpoint()
model.load_state_dict(torch.load(checkpoint, map_location=device))
configure_trainable_layers(model)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
optimizer = optim.AdamW(
    (parameter for parameter in model.parameters() if parameter.requires_grad),
    lr=CFG.LEARNING_RATE,
    weight_decay=CFG.WEIGHT_DECAY,
)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

In [ ]:
def run_epoch(model, loader, optimizer=None):
    """Run one train or evaluation epoch."""
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct += outputs.argmax(dim=1).eq(labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), 100.0 * correct / total


best_acc = -np.inf
patience_count = 0
history = []
best_path = CFG.RESULTS_DIR / "resnet50_ft_v2_best.pth"

for epoch in range(1, CFG.MAX_EPOCHS + 1):
    start_time = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader)
    scheduler.step(val_acc)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": optimizer.param_groups[0]["lr"],
        }
    )
    print(
        f"Epoch {epoch}: val_acc={val_acc:.2f}% "
        f"train_acc={train_acc:.2f}% time={time.time() - start_time:.1f}s"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        patience_count = 0
        torch.save(model.state_dict(), best_path)
    else:
        patience_count += 1
        if patience_count >= CFG.PATIENCE:
            print("Early stopping triggered.")
            break

pd.DataFrame(history).to_csv(CFG.RESULTS_DIR / "resnet50_ft_v2_history.csv", index=False)
print(f"Best validation accuracy: {best_acc:.2f}%")

## 2. Evaluation Handoff

After training, run the same final evaluation protocol from notebook 1
against `resnet50_ft_v2_best.pth`: validation/test top-1 and top-5,
per-class F1, confusion pairs, qualitative errors, and latency. Keep the
metric schema identical so notebook 2 can be compared directly with the
baseline.